# Behavioral Data Analysis

This notebook demonstrates a clean behavioral data analysis workflow using **simulated data**.
The goal is to show how to generate, inspect, clean, summarize, visualize, and statistically test
behavioral data in Python.

All comments and explanatory text are written in English, and variable names are also in English.


In [ ]:
# Import the libraries used throughout the analysis.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.anova import AnovaRM

# Set a random seed so the simulated dataset is reproducible.
np.random.seed(42)


## 1. Simulate a behavioral dataset

Here we create a small repeated-measures dataset with two within-subject conditions:

- `baseline`
- `adaptation`

Each participant has:
- a `switch_rate` value, which can be interpreted as a behavioral measure such as perceptual switch frequency
- a `trials_completed` value, which we later use for a simple quality-control exclusion step


In [ ]:
# Define the number of participants and the experimental conditions.
n_participants = 30
conditions = ["baseline", "adaptation"]

# Store each observation as a dictionary and collect them in a list.
rows = []

# Generate one observation per participant per condition.
for participant_id in range(1, n_participants + 1):
    for condition in conditions:
        # Make the adaptation condition slightly higher on average.
        if condition == "baseline":
            mean_rate = 5.0
        else:
            mean_rate = 6.5

        # Simulate the behavioral outcome.
        switch_rate = np.random.normal(loc=mean_rate, scale=1.0)

        # Simulate the number of completed trials for quality control.
        trials_completed = np.random.randint(80, 121)

        rows.append({
            "participant_id": participant_id,
            "condition": condition,
            "switch_rate": switch_rate,
            "trials_completed": trials_completed
        })

# Convert the simulated observations into a pandas DataFrame.
df = pd.DataFrame(rows)

# Show the first rows of the dataset.
df.head()


## 2. Inspect the dataset

A quick inspection helps confirm that the data structure looks correct before continuing.


In [ ]:
# Check the shape of the dataset.
print("Dataset shape:", df.shape)

# Display summary information about columns and data types.
df.info()

# Show descriptive statistics for the numeric columns.
df.describe()


## 3. Apply a simple quality-control exclusion

In behavioral experiments, it is common to exclude participants who did not complete enough trials.
Here, we exclude participants whose minimum number of completed trials across conditions is below `90`.


In [ ]:
# Set the minimum acceptable number of completed trials.
min_trials = 90

# Find the minimum completed-trial count for each participant.
participant_trial_min = df.groupby("participant_id")["trials_completed"].min()

# Keep only participants who meet the threshold.
valid_participants = participant_trial_min[participant_trial_min >= min_trials].index

# Filter the dataset to retain valid participants only.
df_valid = df[df["participant_id"].isin(valid_participants)].copy()

# Count how many participants were excluded.
n_excluded = n_participants - len(valid_participants)

print(f"Excluded {n_excluded} participant(s) due to insufficient trials.")
print(f"Remaining participants: {len(valid_participants)}")

df_valid.head()


## 4. Compute participant-level summaries

Because this is a within-subject design, it is useful to summarize the data at the participant level.


In [ ]:
# Average switch rate for each participant in each condition.
summary = (
    df_valid
    .groupby(["participant_id", "condition"])["switch_rate"]
    .mean()
    .reset_index()
)

# Reshape the data so each participant has one row and separate columns for each condition.
summary_pivot = summary.pivot(
    index="participant_id",
    columns="condition",
    values="switch_rate"
)

summary_pivot.head()


## 5. Visualize condition means

We plot the mean switch rate for each condition with standard error bars.


In [ ]:
# Compute mean and standard error for each condition.
mean_rates = summary.groupby("condition")["switch_rate"].mean()
sem_rates = summary.groupby("condition")["switch_rate"].sem()

# Create a simple bar chart.
plt.figure(figsize=(6, 4))
plt.bar(mean_rates.index, mean_rates.values, yerr=sem_rates.values, capsize=5)
plt.xlabel("Condition")
plt.ylabel("Mean Switch Rate")
plt.title("Mean Switch Rate by Condition")
plt.tight_layout()
plt.show()


## 6. Run a paired t-test

A paired t-test is appropriate here because the same participants appear in both conditions.


In [ ]:
# Extract the paired observations.
baseline_rates = summary_pivot["baseline"]
adaptation_rates = summary_pivot["adaptation"]

# Run the paired t-test.
t_statistic, p_value = stats.ttest_rel(adaptation_rates, baseline_rates)

print(f"Paired t-test statistic: {t_statistic:.3f}")
print(f"Paired t-test p-value: {p_value:.3f}")


## 7. Run a repeated-measures ANOVA

A repeated-measures ANOVA provides another standard way to test for within-subject condition effects.


In [ ]:
# Fit a repeated-measures ANOVA model.
anova_result = AnovaRM(
    data=summary,
    depvar="switch_rate",
    subject="participant_id",
    within=["condition"]
).fit()

print(anova_result.summary())


## 8. Interpretation

In this simulated dataset, the adaptation condition was designed to have a slightly higher
switch rate than the baseline condition. The statistical tests should usually reflect that pattern,
although exact values may vary if the random seed changes.

This notebook can be adapted easily to real behavioral datasets by replacing the simulation step
with data loading from a `.csv` or similar file.
